# 06 — MEM Electricity | Bronze Ingestion

Capture the Ministry of Energy and Mines (MEM) **Informe de Desempeño — Anexos** workbook as published.

| Item | Value |
|---|---|
| File | `Informe-de-Desempeno-Anexos-<mes>-<año>.xlsx` (monthly; the latest file carries the full history) |
| Sheets captured | `EDE's` (monthly distributor indicators since 2009) and `Nuevo Regimen tarifario` (quarterly tariffs since 2020) |
| Method | One Bronze row per sheet row; all cells kept as a JSON array (dates as ISO text). Layout is interpreted in Silver. |
| Target | `rd_economic_intelligence.bronze.mem_electricity_raw_rows` |

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime, date, timezone
import uuid, json
import openpyxl
from pyspark.sql import functions as F

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/mem_raw/ede_performance/"
TARGET = f"{CATALOG}.bronze.mem_electricity_raw_rows"
SHEETS = ["EDE's", "Nuevo Regimen tarifario"]
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

files = sorted([f for f in dbutils.fs.ls(RAW_PATH) if f.name.lower().endswith(".xlsx")], key=lambda f: f.modificationTime)
if not files:
    raise FileNotFoundError(RAW_PATH)
latest = files[-1]
print("Files in landing:", [f.name for f in files], "· using latest:", latest.name)

def cell(v):
    if isinstance(v, (datetime, date)):
        return v.isoformat()
    return None if v is None else (v if isinstance(v, (int, float)) else str(v))

wb = openpyxl.load_workbook(RAW_PATH + latest.name, data_only=True)
missing = [s for s in SHEETS if s not in wb.sheetnames]
if missing:
    raise ValueError(f"Structural contract failed, sheets missing: {missing}")
rows = []
for s in SHEETS:
    for i, r in enumerate(wb[s].iter_rows(values_only=True), start=1):
        vals = [cell(v) for v in r]
        while vals and vals[-1] is None:
            vals.pop()
        if any(v is not None for v in vals):
            rows.append((s, i, json.dumps(vals, ensure_ascii=False)))

df = (spark.createDataFrame(rows, "_source_sheet string, _source_row_number int, row_json string")
      .withColumn("_source_file", F.lit(latest.name))
      .withColumn("_source_modified_ts", F.lit(datetime.fromtimestamp(latest.modificationTime / 1000, tz=timezone.utc)).cast("timestamp"))
      .withColumn("_ingestion_ts", F.lit(INGESTION_TS).cast("timestamp")).withColumn("_batch_id", F.lit(BATCH_ID)))

if spark.catalog.tableExists(TARGET):
    spark.sql(f"DELETE FROM {TARGET} WHERE _source_file = '{latest.name}'")
df.write.format("delta").mode("append").saveAsTable(TARGET)
spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Bronze: MEM Informe de Desempeño annex rows (EDE monthly indicators and tariff charges), as published.'")
n = spark.table(TARGET).filter(F.col("_batch_id") == BATCH_ID).groupBy("_source_sheet").count().collect()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "file": latest.name, "rows": {r[0]: r[1] for r in n}}, ensure_ascii=False))